In [1]:
# Включаем магию autoreload
# Устанавливаем режим 2: автоматически перезагружать все импортированные модули, если они изменены
%load_ext autoreload
%autoreload 2

In [2]:
import os
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))
print(tf.__version__)

2026-02-05 12:47:28.575213: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-05 12:47:28.598707: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-05 12:47:28.626563: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-05 12:47:28.635141: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-05 12:47:28.656503: I tensorflow/core/platform/cpu_feature_guar

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
2.17.0


I0000 00:00:1770295652.214259    3401 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1770295652.286277    3401 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1770295652.289103    3401 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355


In [3]:
from tqdm.auto import tqdm
import tensorflow as tf
tf.keras.utils.Progbar = tqdm

In [4]:
# import sys
# import pandas as pd

# import os
# import warnings

# warnings.filterwarnings('ignore')
# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
# tf.get_logger().setLevel('ERROR')

# import logging
# logging.getLogger('tensorflow').setLevel(logging.ERROR)

# tf.debugging.set_log_device_placement(False)

In [5]:
import os

# Включает логи XLA. 
# 1 - показывает, что компилируется
# 2 - очень подробные логи
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices' # Обычно не обязательно в 2.17, но для страховки
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '0' # Показать INFO логи


gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU Memory Growth Enabled")
    except RuntimeError as e:
        print(e)

✅ GPU Memory Growth Enabled


In [6]:
# В первой ячейке настройки
from contextlib import contextmanager
import sys
import io
from datetime import datetime
import os

@contextmanager
def training_session(log_name=None):
    """
    Контекстный менеджер для полного логирования сессии обучения
    """
    if log_name is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        log_name = f"training_session_{timestamp}.txt"
    
    log_dir = "training_logs"
    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_name)
    
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    log_file = open(log_path, 'w', encoding='utf-8')
    
    try:
        # Перенаправляем потоки
        sys.stdout = io.TextIOWrapper(io.BufferedWriter(log_file.buffer), encoding='utf-8')
        sys.stderr = sys.stdout
        
        # Выводим заголовок сессии
        print(f"{'='*80}")
        print(f"JUPYTER TRAINING SESSION STARTED")
        print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Log file: {log_path}")
        print(f"{'='*80}\n")
        sys.stdout.flush()
        
        yield log_path
        
    finally:
        # Возвращаем оригинальные потоки
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()
        
        print(f"\n{'='*80}")
        print(f"TRAINING SESSION COMPLETED")
        print(f"Log saved to: {log_path}")
        print(f"{'='*80}")

In [7]:
import os
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import logging
from LSTM_UKF_St_IMM import LSTMIMMUKF
from dataPreparator import HonestDataPreparator
from helpers import optimal_split_for_lstm_ukf

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '0'
tf.get_logger().setLevel('INFO')

# Настройка логирования
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

sys.stdout.flush()

# 1. Загрузка и подготовка OHLC данных
full_df = pd.read_csv('Data/ohlc_df.csv')
# full_df = full_df.iloc[-1000:].copy()
print(f"Загружено {len(full_df)} точек OHLC данных.")

# Инициализация модели
model = LSTMIMMUKF(
    seq_len=72,
    emd_window=120,           # ← уменьшено с 350 до 120
    vol_window=18,            # ← уменьшено с 36 до 18
    vol_window_long=90,       # ← уменьшено с 150 до 90
    min_history_for_features=450,
    save_dir='./checkpoints/model_run',
)

# 2 Подготовка данных БЕЗ утечки будущего (1 вызов!)
train_df, val_df, test_df = model.prepare_honest_datasets(
    full_df=full_df,
    cache_path='./data/honest_prepared',  # Сохранит в ./data/honest_prepared.pkl
    adaptive_blocks=True,          # ← включено
    min_regime_per_block=3,        # ← минимум 4 окна каждого режима в блоке
    max_block_size=250             # ← защита от слишком больших блоков
)

# Логгируем процесс обучения в файл!
# with training_session() as log_path:    
# Обучение
history = model.fit(
    train_df, 
    val_df,
    epochs=15,
    min_epochs=3,
    patience=5,
    save_best_weights=True,
    early_stopping=True
)
    
# Сохранение
print("\n" + "=" * 60)
print("💾 СОХРАНЕНИЕ МОДЕЛИ")
save_path = "saved_models/model_checkpoint"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
model.save(save_path)
print(f"✅ Модель сохранена в: {save_path}")


# # Выполняем оценку
# test_df = test_df.iloc[-500:] # DEBUG

# results = model.evaluate(
#     df=test_df,
#     plot=True,  # Визуализация результатов
#     N=200,  # Количество точек для визуализации
# )

# # Загрузка модели
# print("\n" + "=" * 60)
# print("📥 ЗАГРУЗКА СОХРАНЕННОЙ МОДЕЛИ")
# print("=" * 60)
# # КОРРЕКТНЫЙ СПОСОБ ЗАГРУЗКИ:
# loaded_model = LSTMIMMUKF().load(save_path)

# print("✅ Модель успешно загружена")
# print(f"📊 Счетчик шагов: {loaded_model._step_counter.numpy()}")
# print(f"📊 Состояние инициализировано: {loaded_model._state_initialized.numpy()}")
# print(f"📊 Последнее время скачка: {loaded_model._last_jump_time.numpy()}")

# # Прогнозирование на тестовых данных
# print("\n" + "=" * 60)
# print("🔮 ТЕСТОВОЕ ПРОГНОЗИРОВАНИЕ")
# print("=" * 60)
# # Берем последние 72 точки из валидационных данных как пример
# forecast_input = test_df.iloc[-72:].copy()
# print(f"📊 Используем {len(forecast_input)} точек для прогноза")

# # Выполняем прогноз
# forecast_result = loaded_model.online_predict(forecast_input, return_components=True)
# print("\n✅ РЕЗУЛЬТАТЫ ПРОГНОЗИРОВАНИЯ:")
# print(f"🎯 Прогноз: {forecast_result['level_forecast'][0]:.4f}")
# print(f"📈 95% доверительный интервал: [{forecast_result['level_ci_lower'][0]:.4f}, {forecast_result['level_ci_upper'][0]:.4f}]")
# print(f"🔍 Уровень уверенности: {forecast_result['confidence']:.3f}")
# print(f"📊 Вероятности режимов: {forecast_result['mode_probabilities']}")

print("\n🏁 РАБОТА СКРИПТА ЗАВЕРШЕНА")
sys.stdout.flush()

2026-02-05 12:47:32,872 - INFO - ✅ Reproducibility seed set to 42


Загружено 7000 точек OHLC данных.
🚀 ИНИЦИАЛИЗАЦИЯ LSTM-UKF МОДЕЛИ С КОНТЕКСТНОЙ ВОЛАТИЛЬНОСТЬЮ
✅ Воспроизводимость настроена с seed=42
✅ Обнаружено GPU устройств: 1
✅ Устройство для вычислений: /GPU:0
✅ Инициализированы 21 признаков:
   level, st_comp_diff, log_vol_short, skew, percentile_pos, percentile_pos_fisher, energy, amplitude, yz, gc, p, rs, ht, extreme_pos_momentum, asymmetry_ratio, tail_weight_indicator, velocity, acceleration, rel_vol_short_long, vol_accel_rel, rel_entropy
🔧 UKF режим: DIFFERENTIABLE
✅ Инициализация дифференцируемого UKF...


I0000 00:00:1770295652.875016    3401 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1770295652.877944    3401 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1770295652.880701    3401 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1770295653.036357    3401 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

✅ Дифференцируемый UKF инициализирован для единого режима
✅ Regime Selector инициализирован (LOW/MID/HIGH режимы)
✅ _last_state: shape=(1,)
✅ _last_P: shape=(1, 1, 1) (единая форма [1, 1, 1])
✅ _state_initialized: флаг инициализации
✅ _step_counter: для отслеживания количества шагов
✅ _last_anomaly_time: для отслеживания времени последней аномалии

✅ Инициализация оптимизатора с Loss Scale...
✅ Стандартный оптимизатор инициализирован (без mixed precision)
📁 Директория для сохранения модели: /home/UKF/checkpoints/model_run
✅ Инициализация энтропийного регуляризатора...
✅ Энтропийный регуляризатор инициализирован с lambda=0.0200
✅ LSTM-UKF МОДЕЛЬ С КОНТЕКСТНОЙ ВОЛАТИЛЬНОСТЬЮ ПОЛНОСТЬЮ ИНИЦИАЛИЗИРОВАНА
⏰ Завершено: 2026-02-05 12:47:33
⚠️  Кэш не найден или требуется пересчёт: ./data/honest_prepared.pkl
   Запускаем полную честную подготовку данных (без утечки будущего)...
   • Стратификация: АДАПТИВНАЯ
✅ АДАПТИВНАЯ СТРАТИФИКАЦИЯ ВКЛЮЧЕНА:
   • Мин. окон на режим в блоке: 3
   • Макс. разм

[Parallel(n_jobs=5)]: Using backend LokyBackend with 5 concurrent workers.
2026-02-05 12:47:36.264616: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-05 12:47:36.288804: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-05 12:47:36.296019: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-05 12:47:36.366641: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-05 12:47:36.390571: E external/local_xla/xla/stream_executor/cuda/cuda_d

In [ ]:
# # 8. Возобновление обучения (еще 5 эпох)
# print("\n" + "=" * 60)
# print("🔄 ВОЗОБНОВЛЕНИЕ ОБУЧЕНИЯ (еще 5 эпох)")
# print("=" * 60)
# # Важно: при возобновлении обучения модель автоматически сбрасывает отслеживание лучших весов
# history_resume = loaded_model.fit(
#     train_df, 
#     val_df, 
#     epochs=5,  # продолжаем обучение
#     min_epochs=2,
#     patience=10,
#     save_best_weights=True,
#     early_stopping=True
# )

# # 9. Сохранение дообученной модели
# print("\n" + "=" * 60)
# print("💾 СОХРАНЕНИЕ ДООБУЧЕННОЙ МОДЕЛИ")
# print("=" * 60)
# fine_tuned_path = "saved_models/model_fine_tuned"
# loaded_model.save(fine_tuned_path)
# print(f"✅ Дообученная модель сохранена в: {fine_tuned_path}")

# # 10. Финальное прогнозирование
# print("\n" + "=" * 60)
# print("🎯 ФИНАЛЬНОЕ ПРОГНОЗИРОВАНИЕ")
# print("=" * 60)
# # Используем данные из тестового набора
# test_forecast_input = test_df.iloc[-72:].copy()
# final_forecast = loaded_model.online_predict(test_forecast_input)
# print(f"\n✅ ФИНАЛЬНЫЙ ПРОГНОЗ НА НОВЫХ ДАННЫХ:")
# print(f"📊 Временная метка: {final_forecast['timestamp']}")
# print(f"🎯 Прогноз: {final_forecast['level_forecast'][0]:.4f}")
# print(f"📈 95% доверительный интервал: [{final_forecast['level_ci_lower'][0]:.4f}, {final_forecast['level_ci_upper'][0]:.4f}]")
# print(f"🔍 Уровень уверенности: {final_forecast['confidence']:.3f}")
# print(f"📊 Доминирующий режим: {final_forecast['dominant_mode']}")

# print("\n" + "=" * 60)
# print("🏁 РАБОТА СКРИПТА ЗАВЕРШЕНА")
# print("=" * 60)
# sys.stdout.flush()

In [ ]:
from helpers import optimal_split_for_lstm_ukf

features_df = model.prepare_features(df)

train_df, val_df, test_df = optimal_split_for_lstm_ukf(
    features_df,
    volatility_col='log_vol_short',   # Твоя колонка волатильности
    stratify_col='level',              # Твоя целевая переменная
    train_ratio=0.70,
    val_ratio=0.15,
    window_size=400,  # Экспериментируй: 300-500
    seed=42
)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from Decomposition import get_emd_components
from helpers import text_histogram_formatted, text_histogram_simple

# s, d = get_emd_components(train_df.Close.values)
ax = train_df.level.plot(title='train_df')
text_histogram_simple(train_df.level.values, 20)

# Сохранение графика в файл
plt.savefig('train_df.png')  # сохранит в текущую директорию
plt.show()  # если хотите также показать график

In [ ]:
# s, d = get_emd_components(val_df.Close.values)
ax = val_df.level.plot(title='val_df')
text_histogram_simple(val_df.level.values, 20)

# Сохранение графика в файл
plt.savefig('val_df.png')  # сохранит в текущую директорию
plt.show()  # если хотите также показать график

In [ ]:
# s, d = get_emd_components(test_df.Close.values)
ax = test_df.level.plot(title='test_df')
text_histogram_simple(test_df.level.values, 20)

# Сохранение графика в файл
plt.savefig('test_df.png')  # сохранит в текущую директорию
plt.show()  # если хотите также показать график

In [ ]:
text_histogram_formatted(s)

### Load and Evaluate

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from LSTM_UKF_St_IMM import LSTMIMMUKF, VolatilityRegimeSelector

# 1. Загрузка и подготовка OHLC данных
df = pd.read_csv('Data/ohlc_df.csv')
print(f"Загружено {len(df)} точек OHLC данных.")

# 2. Разделение данных (гарантируем минимум 350 точек для теста)
split_ratio = 0.7
split_idx = int(len(df) * split_ratio)
train_df = df.iloc[:split_idx].copy()
test_val_df = df.iloc[split_idx:].copy()

split_ratio_val = 0.5
split_idx_val = int(len(test_val_df) * split_ratio_val)
val_df = test_val_df.iloc[:split_idx_val].copy()
test_df = test_val_df.iloc[split_idx_val:].copy()

# ⚠️ КРИТИЧЕСКИ ВАЖНО: тест должен содержать минимум 350 точек для расчёта признаков
if len(test_df) < 350:
    raise ValueError(
        f"Недостаточно данных для теста: {len(test_df)} точек. "
        f"Требуется минимум 350 для расчёта признаков (min_history_for_features)."
    )

# For Evaluation - reduce test Dataset
test_df = test_df.iloc[-(450+30):]

print(f"Размер обучающей выборки: {len(train_df)}")
print(f"Размер валидационной выборки: {len(val_df)}")
print(f"Размер тестовой выборки: {len(test_df)}")

# 3. ЗАГРУЗКА МОДЕЛИ — ПРАВИЛЬНЫЙ ПАТТЕРН
print("\n" + "=" * 60)
print("📥 ЗАГРУЗКА СОХРАНЕННОЙ МОДЕЛИ")
print("=" * 60)

# ✅ Способ 1: Создать экземпляр с параметрами → загрузить состояние
# (параметры должны совпадать с обученной моделью)
model = LSTMIMMUKF(
    seq_len=72,
    emd_window=120,           # ← уменьшено с 350 до 120
    vol_window=18,            # ← уменьшено с 36 до 18
    vol_window_long=90,       # ← уменьшено с 150 до 90
    min_history_for_features=450,
    save_dir='./checkpoints/model_run',
)
save_path = "saved_models/model_checkpoint"
model.load(save_path)  # ← загружаем В уже созданный экземпляр

# ✅ Альтернатива: если параметры неизвестны — использовать метаданные
# (реализация ниже)

print("✅ Модель успешно загружена")
print(f"📊 Счетчик шагов: {model._step_counter.numpy()}")
print(f"📊 Состояние инициализировано: {model._state_initialized.numpy()}")
print(f"📊 Время последней аномалии: {model._last_anomaly_time.numpy()}")
print(f"📊 Последнее состояние: {model._last_state.numpy()[:5]}")  # первые 5 значений
print(f"📊 Последняя ковариация: {model._last_P.numpy()[:5]}")    # первые 5 значений

# ⚠️ ВАЖНО: НЕ перезаписывайте состояние вручную!
# Состояние (_last_state, _last_P, _step_counter и др.) уже восстановлено из файла.
# Метод evaluate() сам корректно управляет состоянием:
#   - сохраняет исходное состояние перед оценкой
#   - сбрасывает его для чистого старта оценки
#   - восстанавливает после завершения

# 4. Выполнение оценки (без ручной инициализации состояния!)
print("\n" + "=" * 60)
print("🔍 ЗАПУСК ОЦЕНКИ МОДЕЛИ")
print("=" * 60)
results = model.evaluate(
    df=test_df,
    plot=True,
    N=200,
)

print("\n" + "=" * 60)
print("📊 РЕЗУЛЬТАТЫ ОЦЕНКИ")
print("=" * 60)
for metric, value in results.items():
    print(f"{metric:25s}: {value:.6f}")
print("=" * 60)

### Check Features

In [ ]:
# 1. Загрузка и подготовка OHLC данных
df = pd.read_csv('Data/ohlc_df.csv')
df.head()

In [ ]:
from Decomposition import get_emd_components

st_comp, _ = get_emd_components(df['Close'].values)
df['St_comp'] = st_comp
df['St_comp'].plot()

In [ ]:
perc_window = 100

df['q5_long'] = df['St_comp'].rolling(
    window=perc_window, min_periods=10
).quantile(0.05)
df['q95_long'] = df['St_comp'].rolling(
    window=perc_window, min_periods=10
).quantile(0.95)
df['denom'] = (df['q95_long'] - df['q5_long']).clip(lower=1e-8)
df['percentile_pos'] = (df['St_comp'] - df['q5_long']) / df['denom']

In [ ]:
epsilon = 1e-8
percentile_pos = df['percentile_pos'].values
# percentile_pos_clipped = np.clip(percentile_pos, epsilon, 1.0 - epsilon)
df['percentile_pos_fisher'] = 0.5 * np.log(
(1.0 + percentile_pos_clipped) / (1.0 - percentile_pos_clipped))

print('min value percentile_pos_fisher:', df['percentile_pos_fisher'].min())
print('max value percentile_pos_fisher:', df['percentile_pos_fisher'].max())

In [ ]:
df['percentile_pos_fisher'].isna().sum().sum()

In [ ]:
df['percentile_pos_fisher'].plot()

### Evaluate Saved Model

In [ ]:
# Импортируем класс модели
from LSTM_UKF_St_new import LSTMAttentionUKF

sys.stdout.flush()

# 1. Загрузка и подготовка OHLC данных
df = pd.read_csv('Data/ohlc_df.csv')

print(f"Загружено {len(df)} точек OHLC данных.")

# 2. Разделение данных
split_ratio = 0.7
split_idx = int(len(df) * split_ratio)
train_df = df.iloc[:split_idx].copy()
test_val_df = df.iloc[split_idx:].copy()

split_ratio_val = 0.5
split_idx_val = int(len(test_val_df) * split_ratio_val)
val_df = test_val_df.iloc[:split_idx_val].copy()
test_df = test_val_df.iloc[split_idx_val:].copy()

test_df = test_df.iloc[-360:]

print(f"Размер обучающей выборки: {len(train_df)}")
print(f"Размер валидационной выборки: {len(val_df)}")
print(f"Размер тестовой выборки: {len(test_df)}")

# 2. Инициализация модели с параметрами, соответствующими сохраненной модели
print("\nИнициализация модели...")
model = LSTMAttentionUKF(
    seq_len=36,    # 72
    vol_window=18, # 36
    vol_window_long=350,
    emd_window=200,
    min_history_for_features=150, #350
    seed=42
)

# 3. Загрузка сохраненной модели
print("\nЗагрузка сохраненной модели...")
model_path = './models/best_lstm_ukf/'
try:
    model.load(model_path)
    print("✅ Модель успешно загружена")
except Exception as e:
    print(f"❌ Ошибка при загрузке модели: {str(e)}")
    import traceback
    traceback.print_exc()
    exit(1)

# 4. Убеждаемся, что скейлеры загружены
if model.feature_scalers is None or 'standard' not in model.feature_scalers:
    print("⚠️ Скейлеры не загружены. Выполняем минимум необходимой подготовки...")
    exit(1)

# 5. Запуск оценки модели
print("\n" + "="*80)
print("🚀 ЗАПУСК ОЦЕНКИ МОДЕЛИ")
print("="*80)


# Выполняем оценку
results = model.evaluate(
    test_ohlc=test_df,
    N=300,  # Количество точек для визуализации
    figsize=(14, 12),
    plot=True,  # Визуализация результатов
    save_results=True  # Сохранение результатов
)

In [ ]:
def transform_level(x: np.ndarray, clip_value: float = 3000.0) -> np.ndarray:
    """
    Стабильное преобразование значений St_comp в level с асимметрией для финансовых данных.
    Обеспечивает:
    - Обработку резких движений с разной чувствительностью для роста/падения
    - Численную стабильность для градиентного обучения
    - Обратимость с inverse_transform_level
    
    Args:
        x: Исходные значения St_comp (любого масштаба)
        clip_value: Максимальное абсолютное значение для клиппинга
        
    Returns:
        Преобразованные значения в пространстве уровня
    """
    # 1. Защита от экстремальных значений
    x_clipped = np.clip(x, -clip_value, clip_value)
    
    # 2. Асимметричное преобразование (разные параметры для роста и падения)
    pos_mask = x_clipped >= 0
    neg_mask = x_clipped < 0
    
    # 3. Разные масштабы для позитивных и негативных движений
    #    Рынки обычно имеют асимметричное поведение: падения резче, рост плавнее
    pos_scale = 0.0012  # Меньший масштаб для роста = более плавное преобразование
    neg_scale = 0.0025  # Большой масштаб для падений = более резкое преобразование
    
    # 4. Применение логарифмического преобразования с асимметрией
    result = np.zeros_like(x_clipped)
    
    if np.any(pos_mask):
        # Для положительных значений: более плавное логарифмическое преобразование
        result[pos_mask] = np.log1p(x_clipped[pos_mask] * pos_scale) * 1.8
        
    if np.any(neg_mask):
        # Для отрицательных значений: более резкое преобразование для падений
        result[neg_mask] = -np.log1p(-x_clipped[neg_mask] * neg_scale) * 2.5
    
    # 5. Финальная стабилизация диапазона
    result = np.clip(result, -10.0, 10.0)
    
    return result


@tf.function
def inverse_transform_level(y: tf.Tensor) -> tf.Tensor:
    """
    Стабильная функция восстановления уровня с асимметрией для финансовых данных.
    Обеспечивает:
    - Стабильность градиентов для обучения
    - Асимметричное поведение для роста/падения
    - Контролируемый рост для экстремальных значений
    - Гладкость и дифференцируемость
    
    Args:
        y: Тензор значений уровня в трансформированном пространстве [-inf, +inf]
    Returns:
        Восстановленные значения в исходном масштабе
    """
    # 1. Клиппинг для базовой защиты
    y_clipped = tf.clip_by_value(y, -10.0, 10.0)
    
    # 2. Основная часть с использованием tanh (стабильная для малых значений)
    base_scale = 800.0  # Базовый масштаб для центральной области
    tanh_part = tf.tanh(y_clipped) * base_scale
    
    # 3. Дополнительная логарифмическая часть для экстремальных значений
    #    Асимметричные коэффициенты для роста/падения
    pos_log_factor = 1200.0  # Больший множитель для положительных движений
    neg_log_factor = 600.0   # Меньший множитель для отрицательных движений
    
    #    Логарифмическая компонента с асимметрией
    log_component = tf.where(
        y_clipped >= 0,
        tf.math.log(tf.abs(y_clipped) + 1.0) * pos_log_factor,
        -tf.math.log(tf.abs(y_clipped) + 1.0) * neg_log_factor
    )
    
    # 4. Плавный переход между режимами (в зависимости от абсолютного значения)
    #    При |y| < 1.0 - преимущественно tanh
    #    При |y| > 3.0 - преимущественно логарифмический рост
    transition_weight = tf.sigmoid(tf.abs(y_clipped) - 1.5)  # Плавный переход около |y|=1.5
    
    # 5. Комбинирование компонентов с асимметрией
    result = tanh_part * (1.0 - transition_weight) + log_component * transition_weight
    
    # 6. Финальная стабилизация и ограничение максимального диапазона
    max_abs_value = 3000.0  # Абсолютный максимум для безопасности
    result = tf.clip_by_value(result, -max_abs_value, max_abs_value)
    
    return result    

In [ ]:
a = np.array([1, -0.5, 1000, 100000], dtype=np.float32)

In [ ]:
transformed = transform_level(a)
transformed

In [ ]:
inverse_transform_level(transformed)

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy import stats

def test_transform_functions():
    """
    Комплексный тест для transform_level и inverse_transform_level
    с резко волатильными данными.
    """
    print("=" * 80)
    print("🧪 ТЕСТ ФУНКЦИЙ ПРЕОБРАЗОВАНИЯ УРОВНЯ С ВОЛАТИЛЬНЫМИ ДАННЫМИ")
    print("=" * 80)
    
    # === 1. ГЕНЕРАЦИЯ ТЕСТОВЫХ ДАННЫХ ===
    print("\n" + "-" * 60)
    print("1️⃣ ГЕНЕРАЦИЯ ТЕСТОВЫХ ДАННЫХ")
    print("-" * 60)
    
    # 1.1. Набор 1: Резко волатильная последовательность с асимметрией
    np.random.seed(42)
    t = np.linspace(0, 100, 1000)
    
    # Базовый сигнал с экспоненциальной волатильностью
    volatility = np.exp(0.03 * t) * np.sin(0.1 * t) + 1
    base_signal = np.cumsum(np.random.normal(0, 0.5, 1000))
    
    # Добавляем резкие скачки (асимметричные - больше отрицательных)
    jump_indices = np.random.choice(len(t), size=20, replace=False)
    jump_magnitudes = np.random.exponential(3.0, size=20) * np.where(np.random.rand(20) < 0.7, -1, 0.5)
    
    # Формируем финальный сигнал
    volatile_signal = base_signal.copy()
    for idx, mag in zip(jump_indices, jump_magnitudes):
        volatile_signal[idx:] += mag * volatility[idx]
    
    # Масштабируем для реалистичных значений
    volatile_signal = volatile_signal * 100
    
    # 1.2. Набор 2: Экстремальные значения
    extreme_values = np.array([
        -2500.0, -1500.0, -800.0, -400.0, -200.0, -100.0,
        -50.0, -10.0, -1.0, 0.0,
        1.0, 10.0, 50.0, 100.0, 200.0, 400.0, 800.0, 1500.0, 2500.0, 3000.0
    ])
    
    # 1.3. Набор 3: Реальные рыночные данные (имитация)
    market_data = np.array([
        0.0, 15.3, -8.7, 120.5, -245.8, 56.9, -189.3, 87.2, 321.5, -456.7,
        78.9, -123.4, 234.5, -567.8, 890.1, -1234.5, 1890.3, -2500.0, 300.5, -400.2
    ])
    
    print(f"✅ Сгенерированы 3 набора тестовых данных:")
    print(f"   • Резко волатильная последовательность: {len(volatile_signal)} точек")
    print(f"   • Экстремальные значения: {len(extreme_values)} точек")
    print(f"   • Рыночные данные (имитация): {len(market_data)} точек")
    
    # === 2. ТЕСТИРОВАНИЕ НАБОРОВ ДАННЫХ ===
    test_results = {}
    
    # 2.1. Тест 1: Резко волатильная последовательность
    print("\n" + "-" * 60)
    print("2️⃣ ТЕСТИРОВАНИЕ НАБОРОВ ДАННЫХ")
    print("-" * 60)
    
    def run_test(test_name, test_data):
        """Проводит тест на одном наборе данных и возвращает результаты"""
        print(f"\n📊 Тест: {test_name}")
        print(f"   Диапазон исходных значений: [{np.min(test_data):.2f}, {np.max(test_data):.2f}]")
        print(f"   Стандартное отклонение: {np.std(test_data):.2f}")
        
        # Применяем преобразование и обратное преобразование
        level_transformed = transform_level(test_data.copy())
        level_tf = tf.convert_to_tensor(level_transformed, dtype=tf.float32)
        recovered_values = inverse_transform_level(level_tf).numpy()
        
        # Вычисляем метрики
        mae = np.mean(np.abs(test_data - recovered_values))
        max_error = np.max(np.abs(test_data - recovered_values))
        rmse = np.sqrt(np.mean((test_data - recovered_values) ** 2))
        
        # Проверка на NaN/Inf
        has_nan = np.any(np.isnan(recovered_values))
        has_inf = np.any(np.isinf(recovered_values))
        
        print(f"   MAE восстановления: {mae:.4f}")
        print(f"   Максимальная ошибка: {max_error:.4f}")
        print(f"   RMSE: {rmse:.4f}")
        print(f"   NaN/Inf в восстановленных: {has_nan}/{has_inf}")
        
        return {
            'original': test_data,
            'transformed': level_transformed,
            'recovered': recovered_values,
            'mae': mae,
            'max_error': max_error,
            'rmse': rmse,
            'has_nan': has_nan,
            'has_inf': has_inf
        }
    
    # Запускаем тесты
    test_results['volatile_sequence'] = run_test("Резко волатильная последовательность", volatile_signal)
    test_results['extreme_values'] = run_test("Экстремальные значения", extreme_values)
    test_results['market_data'] = run_test("Рыночные данные (имитация)", market_data)
    
    # === 3. ДЕТАЛЬНЫЙ АНАЛИЗ ЭКСТРЕМАЛЬНЫХ ЗНАЧЕНИЙ ===
    print("\n" + "-" * 60)
    print("3️⃣ ДЕТАЛЬНЫЙ АНАЛИЗ ЭКСТРЕМАЛЬНЫХ ЗНАЧЕНИЙ")
    print("-" * 60)
    
    ext_results = test_results['extreme_values']
    print("\n🔍 Детальный анализ экстремальных значений:")
    print("┌" + "─" * 78 + "┐")
    print(f"│ {'Исходное':^12} │ {'Преобразованное':^18} │ {'Восстановленное':^18} │ {'Ошибка':^12} │ {'Относит. ошибка':^12} │")
    print("├" + "─" * 78 + "┤")
    
    for orig, trans, rec in zip(
        ext_results['original'], 
        ext_results['transformed'], 
        ext_results['recovered']
    ):
        error = abs(orig - rec)
        rel_error = error / (abs(orig) + 1e-8) * 100
        
        # Цветовая индикация ошибки
        error_indicator = "🔴" if error > 100 else ("🟡" if error > 10 else "✅")
        
        print(f"│ {orig:12.2f} │ {trans:18.6f} │ {rec:18.6f} │ {error_indicator} {error:8.2f} │ {rel_error:10.2f}% │")
    
    print("└" + "─" * 78 + "┘")
    
    # === 4. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ ===
    print("\n" + "-" * 60)
    print("4️⃣ ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ")
    print("-" * 60)
    
    plt.figure(figsize=(15, 10))
    
    # 4.1. График 1: Волатильная последовательность
    plt.subplot(2, 2, 1)
    orig = test_results['volatile_sequence']['original']
    rec = test_results['volatile_sequence']['recovered']
    
    plt.plot(orig[:200], label='Исходные значения', alpha=0.8, linewidth=2)
    plt.plot(rec[:200], label='Восстановленные значения', linestyle='--', alpha=0.9, linewidth=2)
    plt.title('Резко волатильная последовательность (первые 200 точек)')
    plt.xlabel('Временной шаг')
    plt.ylabel('Значение')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # 4.2. График 2: Экстремальные значения
    plt.subplot(2, 2, 2)
    orig_ext = test_results['extreme_values']['original']
    rec_ext = test_results['extreme_values']['recovered']
    
    plt.scatter(orig_ext, rec_ext, c='blue', alpha=0.8, s=50)
    plt.plot([np.min(orig_ext), np.max(orig_ext)], [np.min(orig_ext), np.max(orig_ext)], 
             'r--', label='Идеальное восстановление')
    plt.title('Восстановление экстремальных значений')
    plt.xlabel('Исходное значение')
    plt.ylabel('Восстановленное значение')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # 4.3. График 3: Распределение ошибок для волатильной последовательности
    plt.subplot(2, 2, 3)
    errors = np.abs(test_results['volatile_sequence']['original'] - test_results['volatile_sequence']['recovered'])
    plt.hist(errors, bins=50, alpha=0.7, color='green')
    plt.title('Распределение ошибок восстановления')
    plt.xlabel('Абсолютная ошибка')
    plt.ylabel('Частота')
    plt.yscale('log')
    plt.grid(True, alpha=0.3)
    
    # 4.4. График 4: Асимметрия преобразования
    plt.subplot(2, 2, 4)
    test_range = np.linspace(-3000, 3000, 1000)
    transformed = transform_level(test_range.copy())
    tf_tensor = tf.convert_to_tensor(transformed, dtype=tf.float32)
    recovered = inverse_transform_level(tf_tensor).numpy()
    
    # Разная чувствительность для положительных и отрицательных значений
    pos_mask = test_range >= 0
    neg_mask = test_range < 0
    
    plt.plot(test_range[pos_mask], recovered[pos_mask] - test_range[pos_mask], 
             'r-', label='Положительные значения', alpha=0.8)
    plt.plot(test_range[neg_mask], recovered[neg_mask] - test_range[neg_mask], 
             'b-', label='Отрицательные значения', alpha=0.8)
    plt.axhline(y=0, color='k', linestyle='--', alpha=0.5)
    plt.title('Асимметрия ошибки восстановления')
    plt.xlabel('Исходное значение')
    plt.ylabel('Ошибка восстановления')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('transform_functions_test.png', dpi=150, bbox_inches='tight')
    print("✅ Графики сохранены в 'transform_functions_test.png'")
    
    # === 5. СРАВНЕНИЕ С ОРИГИНАЛЬНОЙ ФУНКЦИЕЙ sinh/arcsinh ===
    print("\n" + "-" * 60)
    print("5️⃣ СРАВНЕНИЕ С ОРИГИНАЛЬНОЙ ФУНКЦИЕЙ sinh/arcsinh")
    print("-" * 60)
    
    # Тестовые значения для сравнения
    test_values = np.array([-2500, -1500, -800, -400, -200, -100, -10, 0, 10, 100, 200, 400, 800, 1500, 2500])
    
    def original_transform(x):
        return np.arcsinh(x / 20.0)
    
    def original_inverse_transform(y):
        return np.sinh(y) * 20.0
    
    # Сравнение
    print("\n📊 Сравнение методов восстановления экстремальных значений:")
    print("┌" + "─" * 94 + "┐")
    print(f"│ {'Значение':^10} │ {'Новый метод':^20} │ {'Ошибка (нов.)':^15} │ {'Старый метод':^20} │ {'Ошибка (стар.)':^15} │")
    print("├" + "─" * 94 + "┤")
    
    max_error_new = 0
    max_error_old = 0
    
    for val in test_values:
        # Новый метод
        level_new = transform_level(np.array([val]))[0]
        recovered_new = inverse_transform_level(tf.convert_to_tensor([level_new], dtype=tf.float32)).numpy()[0]
        error_new = abs(val - recovered_new)
        max_error_new = max(max_error_new, error_new)
        
        # Старый метод
        level_old = original_transform(np.array([val]))[0]
        recovered_old = original_inverse_transform(np.array([level_old]))[0]
        error_old = abs(val - recovered_old)
        max_error_old = max(max_error_old, error_old)
        
        # Цветовая индикация
        color_new = '\033[92m' if error_new < 100 else ('\033[93m' if error_new < 500 else '\033[91m')
        color_old = '\033[92m' if error_old < 100 else ('\033[93m' if error_old < 500 else '\033[91m')
        reset = '\033[0m'
        
        print(f"│ {val:10.0f} │ {color_new}{recovered_new:20.2f}{reset} │ {color_new}{error_new:15.2f}{reset} │ {color_old}{recovered_old:20.2f}{reset} │ {color_old}{error_old:15.2f}{reset} │")
    
    print("└" + "─" * 94 + "┘")
    print(f"\n📈 Максимальная ошибка:")
    print(f"   • Новый метод: {max_error_new:.2f}")
    print(f"   • Старый метод: {max_error_old:.2f}")
    
    if max_error_new < max_error_old:
        print("✅ Новый метод показывает меньшую максимальную ошибку")
    else:
        print("⚠️ Старый метод показывает меньшую максимальную ошибку")
    
    # === 6. ИТОГОВЫЙ ВЫВОД ===
    print("\n" + "=" * 80)
    print("📊 ИТОГОВЫЙ ВЫВОД ТЕСТА")
    print("=" * 80)
    
    all_mae = [
        test_results['volatile_sequence']['mae'],
        test_results['extreme_values']['mae'],
        test_results['market_data']['mae']
    ]
    
    all_max_errors = [
        test_results['volatile_sequence']['max_error'],
        test_results['extreme_values']['max_error'],
        test_results['market_data']['max_error']
    ]
    
    print(f"📈 Средняя абсолютная ошибка (MAE) по всем тестам: {np.mean(all_mae):.4f}")
    print(f"📈 Максимальная ошибка по всем тестам: {np.max(all_max_errors):.4f}")
    
    # Проверка критериев стабильности
    stability_passed = True
    
    if np.max(all_max_errors) > 500:
        print("⚠️  КРИТЕРИЙ СТАБИЛЬНОСТИ: Не пройден (макс. ошибка > 500)")
        stability_passed = False
    else:
        print("✅ КРИТЕРИЙ СТАБИЛЬНОСТИ: Пройден (макс. ошибка ≤ 500)")
    
    if any(r['has_nan'] or r['has_inf'] for r in test_results.values()):
        print("⚠️  КРИТЕРИЙ НАДЕЖНОСТИ: Не пройден (обнаружены NaN/Inf)")
        stability_passed = False
    else:
        print("✅ КРИТЕРИЙ НАДЕЖНОСТИ: Пройден (нет NaN/Inf)")
    
    if stability_passed:
        print("\n🎉 ТЕСТ УСПЕШНО ПРОЙДЕН! Функции преобразования готовы к использованию в модели.")
    else:
        print("\n❌ ТЕСТ НЕ ПРОЙДЕН! Требуются доработки функций преобразования.")
    
    print("=" * 80)

# Запуск теста
if __name__ == "__main__":
    # Импортируем функции из основного кода
    # from your_module import transform_level, inverse_transform_level
    
    # Запускаем тест
    test_transform_functions()

In [ ]:
seq_len = 3
def _create_sequences(X: np.ndarray, y: np.ndarray, forecast_horizon=1):
    """
    Создает последовательности для обучения с правильным сдвигом таргета
    forecast_horizon=1 - предсказание следующего шага
    """
    n_samples = len(X) - seq_len - forecast_horizon + 1
    X_seq = np.array([X[i:i + seq_len] for i in range(n_samples)])
    # y_seq содержит следующие значения после окна признаков
    y_seq = np.array([y[i + forecast_horizon:i + seq_len + forecast_horizon] for i in range(n_samples)])
    print('Shapes: ', (X_seq.shape), (y_seq.shape))
    return X_seq, y_seq      

In [ ]:
X = np.array([1,2,3,4,5,6,7,8,9,10])
y = np.array([10,20,30,40,50,60,70,80,90,100])
x_seq, y_seq = _create_sequences(X, y)

print(x_seq)
print(y_seq)

In [ ]:
statistic = model.prepare_features(df).describe().T
statistic

In [ ]:
for el in statistic.iterrows():
    print(el[0],':')
    print(el[1:])
    print()